# Thực nghiệm mở rộng mô hình Hán cổ (guwen-ner · SikuBERT · kraken GPU)

> **Notebook này chỉ chạy trên Google Colab** (colab.research.google.com) — **Runtime → Change runtime type → GPU**, rồi **Run All**. Không chạy trên máy local: mọi `pip install`, `git clone` và inference đều nằm bên trong cell, chỉ thực thi khi mở trên Colab.
>
> Tài liệu nguồn trong repo `family-tree`: [`research/model_survey/EVALUATION.md`](../EVALUATION.md) · [`docs/planning/chinese_genealogy_model_hunt_plan.md`](../../../docs/planning/chinese_genealogy_model_hunt_plan.md)

**Repo:** `family-tree` · nhánh `claude/meeting-project-evaluation-eui68s` · tạo 2026-09-28
**Liên quan:** `docs/planning/chinese_genealogy_model_hunt_plan.md` (§0, §6), `research/model_survey/EVALUATION.md` (mục "Tái lập độc lập 27/9"), `docs/planning/hannom_ner_model_upgrade_proposal.md`.

## Cách dùng
1. Mở trên Google Colab → **Runtime → Change runtime type → GPU** (bắt buộc cho Cell C; A/B chạy được trên CPU).
2. Mỗi cell code **tự chứa**: tự cài dependency, tự tải dữ liệu, tự lưu kết quả vào `/content/results/*.json`. Chạy riêng lẻ cell nào cũng được, không phụ thuộc biến của cell khác.
3. Chạy xong: **File → Download → .ipynb** (giữ output) và push lại vào `research/model_survey/colab/`. Cell cuối nén `/content/results/` để tải JSON kèm theo.

## Nguồn (pin commit / revision để tái lập)
| Thành phần | Nguồn | Pin |
|---|---|---|
| Văn bản phồn thể | [kanripo/KR2a0001](https://github.com/kanripo/KR2a0001) (史記, Kanripo) | `1c19dc6` |
| guwen-ner | [ethanyt/guwen-ner](https://huggingface.co/ethanyt/guwen-ner) · Apache-2.0 | `591273f` |
| GuwenBERT base | [ethanyt/guwenbert-base](https://huggingface.co/ethanyt/guwenbert-base) · Apache-2.0 | `eff0d4a` |
| SikuBERT | [SIKU-BERT/sikubert](https://huggingface.co/SIKU-BERT/sikubert) · license kiểm tra lại trong Cell B | `fc656de` |
| CHAT_models | [colibrisson/CHAT_models](https://github.com/colibrisson/CHAT_models) · CC BY-NC 4.0 (chỉ nghiên cứu) | `9b86ab6` |

## Giới hạn cần nêu khi báo cáo
- **Gold label** (tên người / chức tước) do agent gán tay — cần người rà lại trước khi trích số.
- 史記 gần như chắc chắn nằm trong corpus pretrain của GuwenBERT (殆知阁, đã giản thể hoá) → dữ liệu *in-distribution* về nội dung, **thuận lợi** cho model. Nếu vẫn ra ~0% trên phồn thể thì kết luận càng mạnh; nếu ra cao thì chưa chắc suy rộng sang gia phả Hán-Nôm Việt.
- Đây là 史記 (TQ), không phải gia phả Việt — các đoạn được chọn là đoạn **dạng phả hệ** (X生Y, 父曰…, 其先…, 字…) để gần phong cách gia phả nhất.
- Bản giản thể do **OpenCC t2s** sinh ra (chuyển máy, không phải văn bản giản thể gốc).

In [ ]:
# Cell 0 — Thông tin môi trường (tuỳ chọn, chỉ để ghi lại)
import sys, platform, subprocess
print("python :", sys.version.split()[0])
print("platform:", platform.platform())
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else "Không có GPU — Cell C cần Runtime → Change runtime type → GPU")

## Nhóm A — Falsification check cho guwen-ner (phồn thể vs giản thể)

**Câu hỏi.** Kết quả 27/9 (0/4 thực thể trên *một* câu phồn thể tự soạn, 3/4 trên bản giản thể) có khái quát được không, hay chỉ là hiện tượng của một câu (vd. riêng chữ `諱` → `[UNK]`)?

**Biến độc lập:** dạng chữ (phồn gốc Kanripo / giản do OpenCC t2s). **Biến phụ thuộc:** recall tên người (và chức tước, đếm riêng). **Kiểm soát:** cùng đoạn văn, cùng model + revision, cùng `aggregation_strategy="simple"`, cùng tiêu chí khớp. **Đối chứng dương:** `P0_control` = đúng câu của log 27/9 — kỳ vọng tái lập 0/3 tên (phồn) và 3/3 tên (giản).

**Tiêu chí khớp.** *exact* = span dự đoán trùng nguyên văn gold; *lenient* = span chứa gold (vd. `吳太伯` ⊇ `太伯`) hoặc là chuỗi con ≥2 ký tự của gold. Báo cáo riêng tập tên ≥2 ký tự vì tên 1 chữ (`羽`, `季`, `平`…) khó hơn hẳn.

**Tiêu chí quyết định (đặt trước khi chạy, tính trên P1–P9, recall lenient):**
- **Ủng hộ** "guwen-ner gần như vô dụng trên phồn thể": phồn ≤ 10% **và** giản ≥ 50%.
- **Bác bỏ**: phồn ≥ 50% → kết quả 0/4 hôm 27/9 là hiện tượng riêng của câu đó.
- Ngoài hai vùng trên: **chưa kết luận**, ghi nguyên số.
- **Bổ sung sau smoke test local 28/9 (post-hoc, KHÔNG thuộc tiêu chí đặt trước):** (i) điều kiện **bỏ dấu câu** (2×2: dạng chữ × dấu câu) — câu đối chứng P0 không có dấu câu còn đoạn 史記 có, và gia phả OCR thật thường không có dấu câu; (ii) đếm **span chứa ký tự `[UNK]` ngoài gold** (span dự đoán có ký tự `[UNK]` và không trùng tên/chức gold nào). Chỉ là chỉ báo nhiễu thô: vì chưa gán nhãn địa danh, cột này gồm cả địa danh đúng (`魯`, `隴西`…) lẫn nhiễu thật (`時`, `為將`, `讀`, `書`…) → **phải rà tay** cột danh sách span; không phải precision.
- **Cơ chế:** nếu các đoạn phồn thể thất bại đồng thời có tỷ lệ `[UNK]` cao → nguyên nhân là độ phủ vocab (vocab GuwenBERT là giản thể). Nếu `[UNK]` thấp mà vẫn thất bại → cơ chế khác, cần điều tra thêm.

In [ ]:
# === Cell A — guwen-ner trên đoạn 史記 thật: phồn thể vs giản thể ===
import subprocess, sys
TRANSFORMERS_PIN = "4.46.3"   # khớp log 26–27/9; đặt "" để dùng bản Colab có sẵn nếu pip lỗi
if "transformers" in sys.modules:
    print("⚠️ transformers đã được import trong runtime này — nếu khác version pin, hãy Runtime → Restart session rồi chạy lại cell.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                f"transformers=={TRANSFORMERS_PIN}" if TRANSFORMERS_PIN else "transformers",
                "opencc-python-reimplemented"], check=True)

import json, re, math, datetime, platform, pathlib, urllib.request
from importlib.metadata import version

# ---- Bộ test (lặp nguyên văn ở Cell A và Cell B — cố ý, để mỗi cell tự chứa) ----
KANRIPO_REPO = "kanripo/KR2a0001"   # 史記 — Kanripo, base edition "tls", chữ phồn thể
KANRIPO_SHA = "1c19dc6fa970b1c530fced9e8e3697d19163c26c"
# per   = tên người; title = chức/tước (đếm riêng).
# Gold do agent gán tay 2026-09-28 — CẦN NGƯỜI RÀ LẠI trước khi trích số vào luận văn.
TESTSET = [
    {"id": "P0_control", "chapter": "câu tự soạn 26–27/9 (đối chứng dương, KHÔNG phải nguồn thật)", "file": None,
     "text": "始祖諱文成官至知府娶阮氏生子一人諱德重", "per": ["文成", "阮氏", "德重"], "title": ["知府"]},
    {"id": "P1_kongzi", "chapter": "孔子世家", "file": "KR2a0001_400.txt",
     "text": "孔子生魯昌平鄉陬邑。其先宋人也，曰孔防叔。防叔生伯夏，伯夏生叔梁紇。紇與顏氏女野合而生孔子",
     "per": ["孔子", "孔防叔", "伯夏", "叔梁紇", "顏氏"], "title": []},
    {"id": "P2_taibo", "chapter": "吳太伯世家", "file": "KR2a0001_400.txt",
     "text": "吳太伯，太伯弟仲雍，皆周太王之子，而王季歷之兄也。季歷賢，而有聖子昌",
     "per": ["太伯", "仲雍", "太王", "季歷", "昌"], "title": []},
    {"id": "P3_xiangyu", "chapter": "項羽本紀", "file": "KR2a0001_100.txt",
     "text": "項籍者，下相人也，字羽。初起時，年二十四。其季父項梁，梁父即楚將項燕，為秦將王翦所戮者也。",
     "per": ["項籍", "羽", "項梁", "項燕", "王翦"], "title": []},
    {"id": "P4_gaozu", "chapter": "高祖本紀", "file": "KR2a0001_100.txt",
     "text": "高祖，沛豐邑中陽里人，姓劉氏，字季。父曰太公，母曰劉媼。",
     "per": ["高祖", "季", "太公", "劉媼"], "title": []},
    {"id": "P5_laozi", "chapter": "老子韓非列傳", "file": "KR2a0001_500.txt",
     "text": "老子者，楚苦縣厲鄉曲仁里人也，姓李氏，名耳，字聃，周守藏室之史也。",
     "per": ["老子", "耳", "聃"], "title": ["守藏室之史"]},
    {"id": "P6_mengchang", "chapter": "孟嘗君列傳", "file": "KR2a0001_500.txt",
     "text": "孟嘗君名文，姓田氏。文之父曰靖郭君田嬰，田嬰者，齊威王少子而齊宣王庶弟也。",
     "per": ["孟嘗君", "文", "田嬰", "齊威王", "齊宣王"], "title": ["靖郭君"]},
    {"id": "P7_zhangliang", "chapter": "留侯世家", "file": "KR2a0001_400.txt",
     "text": "留侯張良者，其先韓人也。大父開地，相韓昭侯、宣惠王、襄哀王。父平，相釐王、悼惠王。",
     "per": ["張良", "開地", "韓昭侯", "宣惠王", "襄哀王", "平", "釐王", "悼惠王"], "title": ["留侯"]},
    {"id": "P8_liguang", "chapter": "李將軍列傳", "file": "KR2a0001_500.txt",
     "text": "李將軍廣者，隴西成紀人也。其先曰李信，秦時為將，逐得燕太子丹者也。",
     "per": ["廣", "李信", "太子丹"], "title": ["將軍"]},
    {"id": "P9_chenping", "chapter": "陳丞相世家", "file": "KR2a0001_400.txt",
     "text": "陳丞相平者，陽武戶牖鄉人也。少時家貧，好讀書，有田三十畝，獨與兄伯居。",
     "per": ["平", "伯"], "title": ["丞相"]},
]


def load_kanripo(fname):
    url = f"https://raw.githubusercontent.com/{KANRIPO_REPO}/{KANRIPO_SHA}/{fname}"
    raw = urllib.request.urlopen(url, timeout=60).read().decode("utf-8")
    body = "\n".join(l for l in raw.splitlines() if not l.startswith(("#", "*")))
    body = re.sub(r"<pb:[^>]*>", "", body)
    return re.sub(r"[¶\s\u3000]", "", body)


PUNCT = re.compile(r"[，。、；：「」『』？！（）《》〈〉·]")


def build_testset():
    from opencc import OpenCC
    cc = OpenCC("t2s")
    cache, items = {}, []
    for src in TESTSET:
        it = dict(src)
        for g in it["per"] + it["title"]:
            assert g in it["text"], f"{it['id']}: gold {g!r} không có trong text"
        if it["file"]:
            if it["file"] not in cache:
                cache[it["file"]] = load_kanripo(it["file"])
            assert it["text"] in cache[it["file"]], (
                f"{it['id']}: đoạn trích KHÔNG có nguyên văn trong {it['file']}@{KANRIPO_SHA[:7]}")
            it["source_url"] = f"https://github.com/{KANRIPO_REPO}/blob/{KANRIPO_SHA}/{it['file']}"
        simp = cc.convert(it["text"])
        assert len(simp) == len(it["text"]), f"{it['id']}: OpenCC t2s đổi độ dài chuỗi"
        it["trad"] = {"text": it["text"], "per": it["per"], "title": it["title"]}
        it["simp"] = {"text": simp, "per": [cc.convert(g) for g in it["per"]],
                      "title": [cc.convert(g) for g in it["title"]]}
        for key in ("trad", "simp"):
            d = it[key]
            it[key + "_nopunct"] = {"text": PUNCT.sub("", d["text"]), "per": d["per"], "title": d["title"]}
        items.append(it)
    return items


import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

MODEL_ID, MODEL_REV = "ethanyt/guwen-ner", "591273ff61bb5eb453f47c300f577058d6d2ee15"
items = build_testset()
p0 = next(it for it in items if it["id"] == "P0_control")
print("P0 giản thể (OpenCC):", p0["simp"]["text"],
      "| khớp câu giản thể log 27/9:", p0["simp"]["text"] == "始祖讳文成官至知府娶阮氏生子一人讳德重")

tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REV)
mdl = AutoModelForTokenClassification.from_pretrained(MODEL_ID, revision=MODEL_REV)
ner = pipeline("token-classification", model=mdl, tokenizer=tok, aggregation_strategy="simple",
               device=0 if torch.cuda.is_available() else -1)
print("labels:", sorted(set(mdl.config.id2label.values())))


def run_ner(text):
    out = []
    for e in ner(text):
        s, t = e.get("start"), e.get("end")
        surf = text[s:t] if s is not None and t is not None else e["word"].replace(" ", "")
        out.append({"surface": surf, "label": e["entity_group"], "score": round(float(e["score"]), 4)})
    return out


def unk_chars(text):
    return [ch for ch in text if tok.unk_token_id in tok(ch, add_special_tokens=False)["input_ids"]]


def hits(gold, surfaces):
    exact = [g for g in gold if g in surfaces]
    lenient = [g for g in gold if any(g == p or g in p or (len(p) >= 2 and p in g) for p in surfaces)]
    return exact, lenient


SCRIPTS = ("trad", "simp", "trad_nopunct", "simp_nopunct")
LABEL = {"trad": "phồn", "simp": "giản", "trad_nopunct": "phồn, bỏ dấu", "simp_nopunct": "giản, bỏ dấu"}
rows = []
for it in items:
    for script in SCRIPTS:
        d = it[script]
        preds = run_ner(d["text"])
        surf = [p["surface"] for p in preds]
        pe, pl = hits(d["per"], surf)
        multi = [g for g in d["per"] if len(g) >= 2]
        _, ml = hits(multi, surf)
        _, tl = hits(d["title"], surf)
        unk = unk_chars(d["text"])
        gold_all = d["per"] + d["title"]
        spurious_unk = [p for p in surf if any(ch in unk for ch in p)
                        and not any(g in p or p in g for g in gold_all)]
        rows.append({
            "id": it["id"], "script": script, "text": d["text"],
            "n_chars": len(d["text"]), "unk_n": len(unk), "unk_chars": "".join(unk),
            "n_per": len(d["per"]), "per_exact": len(pe), "per_lenient": len(pl),
            "n_per_multi": len(multi), "per_multi_lenient": len(ml),
            "n_title": len(d["title"]), "title_lenient": len(tl),
            "n_pred": len(preds), "spurious_unk_spans": spurious_unk, "n_spurious_unk": len(spurious_unk),
            "missed_per": [g for g in d["per"] if g not in pl],
            "preds": preds,
        })

real = [r for r in rows if r["id"] != "P0_control"]


def micro(script, hit, n):
    rs = [r for r in real if r["script"] == script]
    den = sum(r[n] for r in rs)
    return (sum(r[hit] for r in rs) / den if den else float("nan")), sum(r[hit] for r in rs), den


summary = {}
for script in SCRIPTS:
    summary[script] = {
        "per_exact": micro(script, "per_exact", "n_per"),
        "per_lenient": micro(script, "per_lenient", "n_per"),
        "per_multi_lenient": micro(script, "per_multi_lenient", "n_per_multi"),
        "title_lenient": micro(script, "title_lenient", "n_title"),
        "unk_rate": micro(script, "unk_n", "n_chars"),
        "spurious_unk_per_pred": micro(script, "n_spurious_unk", "n_pred"),
    }

per_passage = {}
for r in (r for r in real if r["script"] in ("trad", "simp")):
    per_passage.setdefault(r["id"], {})[r["script"]] = r["per_lenient"] / r["n_per"]
wins = sum(v["simp"] > v["trad"] for v in per_passage.values())
losses = sum(v["simp"] < v["trad"] for v in per_passage.values())
ties = len(per_passage) - wins - losses
n = wins + losses
sign_p = min(1.0, 2 * sum(math.comb(n, k) for k in range(max(wins, losses), n + 1)) / 2 ** n) if n else float("nan")

pct = lambda a, b: f"{a}/{b} ({a / b:.0%})" if b else "—"
row = lambda i, sc: next(r for r in rows if r["id"] == i and r["script"] == sc)
print("\n### Bảng theo đoạn (recall tên người, lenient)\n")
print("| id | " + " | ".join(LABEL[sc] for sc in SCRIPTS) + " | [UNK] (phồn) | span chứa [UNK] ngoài gold (phồn) |")
print("|---" * (len(SCRIPTS) + 3) + "|")
for it in items:
    cells = [pct(row(it["id"], sc)["per_lenient"], row(it["id"], sc)["n_per"]) for sc in SCRIPTS]
    t = row(it["id"], "trad")
    print(f"| {it['id']} | " + " | ".join(cells) + f" | {t['unk_n']}/{t['n_chars']} {t['unk_chars'] or ''} | "
          f"{' '.join(t['spurious_unk_spans']) or '—'} |")

print("\n### Tổng hợp P1–P9 (micro)\n")
print("| chỉ số | " + " | ".join(LABEL[sc] for sc in SCRIPTS) + " |")
print("|---" * (len(SCRIPTS) + 1) + "|")
for k, label in [("per_exact", "tên người — exact"), ("per_lenient", "tên người — lenient"),
                 ("per_multi_lenient", "tên ≥2 chữ — lenient"), ("title_lenient", "chức tước — lenient"),
                 ("unk_rate", "tỷ lệ ký tự [UNK]"), ("spurious_unk_per_pred", "span chứa [UNK] ngoài gold / tổng span")]:
    print(f"| {label} | " + " | ".join(f"{summary[sc][k][1]}/{summary[sc][k][2]} ({summary[sc][k][0]:.1%})" for sc in SCRIPTS) + " |")
print(f"\nSign test theo đoạn (giản > phồn): thắng {wins}, thua {losses}, hoà {ties}; p hai phía = {sign_p:.4f} (n nhỏ — chỉ tham khảo)")

t0, s0 = row("P0_control", "trad"), row("P0_control", "simp")
print(f"\nĐối chứng dương P0: phồn {t0['per_lenient']}/3, giản {s0['per_lenient']}/3 "
      f"→ {'TÁI LẬP log 27/9' if (t0['per_lenient'], s0['per_lenient']) == (0, 3) else 'KHÁC log 27/9 — kiểm tra môi trường trước khi đọc kết quả P1–P9'}")

def decide(tr, sr):
    if tr >= 0.5:
        return "BÁC BỎ 'vô dụng trên phồn thể' (phồn ≥ 50%)"
    if tr <= 0.10 and sr >= 0.5:
        return "ỦNG HỘ 'gần như vô dụng trên phồn thể'"
    return "CHƯA KẾT LUẬN — báo cáo nguyên số"


verdict = decide(summary["trad"]["per_lenient"][0], summary["simp"]["per_lenient"][0])
verdict_nopunct = decide(summary["trad_nopunct"]["per_lenient"][0], summary["simp_nopunct"]["per_lenient"][0])
print("Kết luận theo tiêu chí đặt trước (có dấu câu):", verdict)
print("Cùng tiêu chí, điều kiện bỏ dấu câu (post-hoc):", verdict_nopunct)
fail = [r for r in real if r["script"] == "trad" and r["per_lenient"] == 0]
ok = [r for r in real if r["script"] == "trad" and r["per_lenient"] > 0]
mean = lambda xs: sum(xs) / len(xs) if xs else float("nan")
print(f"Cơ chế: tỷ lệ [UNK] trung bình ở đoạn phồn thể thất bại = {mean([r['unk_n'] / r['n_chars'] for r in fail]):.1%} "
      f"(n={len(fail)}), ở đoạn có bắt được ≥1 tên = {mean([r['unk_n'] / r['n_chars'] for r in ok]):.1%} (n={len(ok)})")

out = pathlib.Path("/content/results"); out.mkdir(parents=True, exist_ok=True)
json.dump({
    "experiment": "A_guwen_ner_trad_vs_simp",
    "timestamp_utc": datetime.datetime.utcnow().isoformat() + "Z",
    "env": {"python": platform.python_version(), "torch": torch.__version__, "transformers": version("transformers"),
            "opencc": version("opencc-python-reimplemented"),
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},
    "model": {"id": MODEL_ID, "revision": MODEL_REV},
    "source": {"repo": KANRIPO_REPO, "sha": KANRIPO_SHA},
    "criteria": {"support": "trad<=10% and simp>=50%", "reject": "trad>=50%", "metric": "micro lenient PER recall P1-P9"},
    "summary": summary, "sign_test": {"wins": wins, "losses": losses, "ties": ties, "p_two_sided": sign_p},
    "verdict": verdict, "verdict_nopunct_posthoc": verdict_nopunct, "rows": rows,
}, open(out / "A_guwen_ner.json", "w"), ensure_ascii=False, indent=1)
print("\nĐã lưu /content/results/A_guwen_ner.json")

## Nhóm B — SikuBERT (lần đầu thử) so với GuwenBERT

**Bước 1 — kiểm tra license (bắt buộc theo §0.3 của plan).** Kiểm tra trước khi viết notebook (2026-09-28): model card HF `SIKU-BERT/sikubert` khai `license: apache-2.0` (tag `license:apache-2.0`, revision `fc656de`); repo GitHub [hsc748NLP/SikuBERT-…](https://github.com/hsc748NLP/SikuBERT-for-digital-humanities-and-classical-Chinese-information-processing) cũng là Apache-2.0. Cell dưới **kiểm tra lại trực tiếp** qua HF API và **dừng, không tải weight** nếu license không thuộc danh sách cho phép.

**Bước 2 — phép thử.** SikuBERT chỉ là model pretrain (fill-mask), **không có head NER** → *không thể* so recall NER trực tiếp với guwen-ner nếu không fine-tune (ngoài phạm vi). Thay vào đó đo **độ phủ vocab và khả năng hiểu ngữ cảnh** trên cùng bộ đoạn của Cell A, so với `ethanyt/guwenbert-base` (model nền của guwen-ner):
- Che (mask) từng ký tự Hán, tính top-1 / top-5 đúng. Tách **ký tự thuộc tên người/chức tước** (`ent`) và **ký tự còn lại** (`ctrl`, mốc nền năng lực ngôn ngữ).
- Ký tự bị tokenizer biến thành `[UNK]` tính là trượt (và đếm riêng).

**Dự đoán:** SikuBERT học trên 四库全书 (phồn thể) → khoảng cách phồn/giản nhỏ; GuwenBERT (corpus đã giản thể hoá) → phồn thể kém hơn rõ. Nếu đúng: SikuBERT là nền hợp lý hơn để fine-tune NER cho gia phả Hán-Nôm (vốn là phồn thể). Nếu GuwenBERT không tụt trên phồn thể ở phép thử này → giả thuyết "thiếu vocab phồn thể" không giải thích được thất bại của guwen-ner.

In [ ]:
# === Cell B — license gate + fill-mask: SikuBERT vs GuwenBERT, phồn vs giản ===
import subprocess, sys
TRANSFORMERS_PIN = "4.46.3"
if "transformers" in sys.modules:
    print("⚠️ transformers đã được import trong runtime này — nếu khác version pin, hãy Runtime → Restart session rồi chạy lại cell.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                f"transformers=={TRANSFORMERS_PIN}" if TRANSFORMERS_PIN else "transformers",
                "opencc-python-reimplemented"], check=True)

import json, re, datetime, platform, pathlib, urllib.request
from importlib.metadata import version

MODELS = {
    "SIKU-BERT/sikubert": "fc656de2d6bde33919102dd3abe31c843f42226a",
    "ethanyt/guwenbert-base": "eff0d4a5196d7bf7b8be746c5c6437e89d8b9061",
}
ALLOWED = {"apache-2.0", "mit", "bsd", "bsd-2-clause", "bsd-3-clause"}
license_check = {}
for mid in MODELS:
    info = json.load(urllib.request.urlopen(f"https://huggingface.co/api/models/{mid}", timeout=60))
    card = (info.get("cardData") or {}).get("license")
    tags = [t.split(":", 1)[1] for t in info.get("tags", []) if t.startswith("license:")]
    lic = {x.lower() for x in ([card] if card else []) + tags}
    license_check[mid] = {"card": card, "tags": tags, "ok": bool(lic) and lic <= ALLOWED, "head_sha": info.get("sha")}
    print(f"{mid}: card={card!r} tags={tags} head_sha={info.get('sha')} → {'OK' if license_check[mid]['ok'] else 'KHÔNG HỢP LỆ'}")
bad = [m for m, v in license_check.items() if not v["ok"]]
if bad:
    raise SystemExit(f"Dừng: license không thuộc {sorted(ALLOWED)} hoặc không khai báo: {bad}. Không tải weight.")

# ---- Bộ test (lặp nguyên văn ở Cell A và Cell B — cố ý, để mỗi cell tự chứa) ----
KANRIPO_REPO = "kanripo/KR2a0001"   # 史記 — Kanripo, base edition "tls", chữ phồn thể
KANRIPO_SHA = "1c19dc6fa970b1c530fced9e8e3697d19163c26c"
# per   = tên người; title = chức/tước (đếm riêng).
# Gold do agent gán tay 2026-09-28 — CẦN NGƯỜI RÀ LẠI trước khi trích số vào luận văn.
TESTSET = [
    {"id": "P0_control", "chapter": "câu tự soạn 26–27/9 (đối chứng dương, KHÔNG phải nguồn thật)", "file": None,
     "text": "始祖諱文成官至知府娶阮氏生子一人諱德重", "per": ["文成", "阮氏", "德重"], "title": ["知府"]},
    {"id": "P1_kongzi", "chapter": "孔子世家", "file": "KR2a0001_400.txt",
     "text": "孔子生魯昌平鄉陬邑。其先宋人也，曰孔防叔。防叔生伯夏，伯夏生叔梁紇。紇與顏氏女野合而生孔子",
     "per": ["孔子", "孔防叔", "伯夏", "叔梁紇", "顏氏"], "title": []},
    {"id": "P2_taibo", "chapter": "吳太伯世家", "file": "KR2a0001_400.txt",
     "text": "吳太伯，太伯弟仲雍，皆周太王之子，而王季歷之兄也。季歷賢，而有聖子昌",
     "per": ["太伯", "仲雍", "太王", "季歷", "昌"], "title": []},
    {"id": "P3_xiangyu", "chapter": "項羽本紀", "file": "KR2a0001_100.txt",
     "text": "項籍者，下相人也，字羽。初起時，年二十四。其季父項梁，梁父即楚將項燕，為秦將王翦所戮者也。",
     "per": ["項籍", "羽", "項梁", "項燕", "王翦"], "title": []},
    {"id": "P4_gaozu", "chapter": "高祖本紀", "file": "KR2a0001_100.txt",
     "text": "高祖，沛豐邑中陽里人，姓劉氏，字季。父曰太公，母曰劉媼。",
     "per": ["高祖", "季", "太公", "劉媼"], "title": []},
    {"id": "P5_laozi", "chapter": "老子韓非列傳", "file": "KR2a0001_500.txt",
     "text": "老子者，楚苦縣厲鄉曲仁里人也，姓李氏，名耳，字聃，周守藏室之史也。",
     "per": ["老子", "耳", "聃"], "title": ["守藏室之史"]},
    {"id": "P6_mengchang", "chapter": "孟嘗君列傳", "file": "KR2a0001_500.txt",
     "text": "孟嘗君名文，姓田氏。文之父曰靖郭君田嬰，田嬰者，齊威王少子而齊宣王庶弟也。",
     "per": ["孟嘗君", "文", "田嬰", "齊威王", "齊宣王"], "title": ["靖郭君"]},
    {"id": "P7_zhangliang", "chapter": "留侯世家", "file": "KR2a0001_400.txt",
     "text": "留侯張良者，其先韓人也。大父開地，相韓昭侯、宣惠王、襄哀王。父平，相釐王、悼惠王。",
     "per": ["張良", "開地", "韓昭侯", "宣惠王", "襄哀王", "平", "釐王", "悼惠王"], "title": ["留侯"]},
    {"id": "P8_liguang", "chapter": "李將軍列傳", "file": "KR2a0001_500.txt",
     "text": "李將軍廣者，隴西成紀人也。其先曰李信，秦時為將，逐得燕太子丹者也。",
     "per": ["廣", "李信", "太子丹"], "title": ["將軍"]},
    {"id": "P9_chenping", "chapter": "陳丞相世家", "file": "KR2a0001_400.txt",
     "text": "陳丞相平者，陽武戶牖鄉人也。少時家貧，好讀書，有田三十畝，獨與兄伯居。",
     "per": ["平", "伯"], "title": ["丞相"]},
]


def load_kanripo(fname):
    url = f"https://raw.githubusercontent.com/{KANRIPO_REPO}/{KANRIPO_SHA}/{fname}"
    raw = urllib.request.urlopen(url, timeout=60).read().decode("utf-8")
    body = "\n".join(l for l in raw.splitlines() if not l.startswith(("#", "*")))
    body = re.sub(r"<pb:[^>]*>", "", body)
    return re.sub(r"[¶\s\u3000]", "", body)


PUNCT = re.compile(r"[，。、；：「」『』？！（）《》〈〉·]")


def build_testset():
    from opencc import OpenCC
    cc = OpenCC("t2s")
    cache, items = {}, []
    for src in TESTSET:
        it = dict(src)
        for g in it["per"] + it["title"]:
            assert g in it["text"], f"{it['id']}: gold {g!r} không có trong text"
        if it["file"]:
            if it["file"] not in cache:
                cache[it["file"]] = load_kanripo(it["file"])
            assert it["text"] in cache[it["file"]], (
                f"{it['id']}: đoạn trích KHÔNG có nguyên văn trong {it['file']}@{KANRIPO_SHA[:7]}")
            it["source_url"] = f"https://github.com/{KANRIPO_REPO}/blob/{KANRIPO_SHA}/{it['file']}"
        simp = cc.convert(it["text"])
        assert len(simp) == len(it["text"]), f"{it['id']}: OpenCC t2s đổi độ dài chuỗi"
        it["trad"] = {"text": it["text"], "per": it["per"], "title": it["title"]}
        it["simp"] = {"text": simp, "per": [cc.convert(g) for g in it["per"]],
                      "title": [cc.convert(g) for g in it["title"]]}
        for key in ("trad", "simp"):
            d = it[key]
            it[key + "_nopunct"] = {"text": PUNCT.sub("", d["text"]), "per": d["per"], "title": d["title"]}
        items.append(it)
    return items


import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

device = "cuda" if torch.cuda.is_available() else "cpu"
items = build_testset()
real = [it for it in items if it["id"] != "P0_control"]
CJK = re.compile(r"[\u3400-\u9fff\uf900-\ufaff\U00020000-\U0003134f]")


def entity_positions(text, golds):
    pos = set()
    for g in golds:
        for m in re.finditer(re.escape(g), text):
            pos.update(range(m.start(), m.end()))
    return pos


def probe(tok, mlm, text, golds):
    enc = tok(text, return_offsets_mapping=True, return_tensors="pt")
    ids = enc["input_ids"][0]
    char2tok = {s: i for i, (s, e) in enumerate(enc["offset_mapping"][0].tolist()) if e - s == 1}
    ent = entity_positions(text, golds)
    stats = {k: {"n": 0, "top1": 0, "top5": 0, "unk_target": 0, "no_token": 0} for k in ("ent", "ctrl")}
    batch, meta = [], []
    for ci, ch in enumerate(text):
        if not CJK.match(ch):
            continue
        kind = "ent" if ci in ent else "ctrl"
        stats[kind]["n"] += 1
        ti = char2tok.get(ci)
        if ti is None:
            stats[kind]["no_token"] += 1
            continue
        if ids[ti].item() == tok.unk_token_id:
            stats[kind]["unk_target"] += 1
            continue
        x = ids.clone()
        x[ti] = tok.mask_token_id
        batch.append(x)
        meta.append((ci, ti, kind))
    for s in range(0, len(batch), 64):
        X = torch.stack(batch[s:s + 64]).to(device)
        with torch.no_grad():
            logits = mlm(input_ids=X, attention_mask=torch.ones_like(X)).logits
        for row, (ci, ti, kind) in enumerate(meta[s:s + 64]):
            top = tok.convert_ids_to_tokens(logits[row, ti].topk(5).indices.tolist())
            stats[kind]["top1"] += int(top[0] == text[ci])
            stats[kind]["top5"] += int(text[ci] in top)
    return stats


results, per_passage = {}, []
for mid, rev in MODELS.items():
    tok = AutoTokenizer.from_pretrained(mid, revision=rev)
    mlm = AutoModelForMaskedLM.from_pretrained(mid, revision=rev).to(device).eval()
    assert tok.is_fast, f"{mid}: cần fast tokenizer để lấy offset"
    for script in ("trad", "simp"):
        agg = {k: {"n": 0, "top1": 0, "top5": 0, "unk_target": 0, "no_token": 0} for k in ("ent", "ctrl")}
        for it in real:
            d = it[script]
            st = probe(tok, mlm, d["text"], d["per"] + d["title"])
            per_passage.append({"model": mid, "script": script, "id": it["id"], **{f"{k}_{m}": st[k][m] for k in st for m in st[k]}})
            for k in agg:
                for m in agg[k]:
                    agg[k][m] += st[k][m]
        results[(mid, script)] = agg
    del mlm
    if device == "cuda":
        torch.cuda.empty_cache()

pct = lambda a, b: f"{a}/{b} ({a / b:.1%})" if b else "—"
print("\n### Fill-mask P1–P9 (micro)\n")
print("| model | chữ | tên/chức top-1 | tên/chức top-5 | nền top-1 | nền top-5 | ký tự → [UNK] |")
print("|---|---|---|---|---|---|---|")
for (mid, script), a in results.items():
    unk = a["ent"]["unk_target"] + a["ctrl"]["unk_target"]
    tot = a["ent"]["n"] + a["ctrl"]["n"]
    print(f"| {mid} | {script} | {pct(a['ent']['top1'], a['ent']['n'])} | {pct(a['ent']['top5'], a['ent']['n'])} | "
          f"{pct(a['ctrl']['top1'], a['ctrl']['n'])} | {pct(a['ctrl']['top5'], a['ctrl']['n'])} | {pct(unk, tot)} |")

print("\nKhoảng cách giản − phồn (nền top-1, điểm %):")
gaps = {}
for mid in MODELS:
    r = lambda s: results[(mid, s)]["ctrl"]["top1"] / max(1, results[(mid, s)]["ctrl"]["n"])
    gaps[mid] = r("simp") - r("trad")
    print(f"  {mid}: {gaps[mid] * 100:+.1f}")
print("Lưu ý: đây là phép thử fill-mask, KHÔNG phải recall NER — không đặt cạnh số của Cell A như cùng thang đo.")

out = pathlib.Path("/content/results"); out.mkdir(parents=True, exist_ok=True)
json.dump({
    "experiment": "B_sikubert_vs_guwenbert_fillmask",
    "timestamp_utc": datetime.datetime.utcnow().isoformat() + "Z",
    "env": {"python": platform.python_version(), "torch": torch.__version__, "transformers": version("transformers"),
            "opencc": version("opencc-python-reimplemented"),
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},
    "license_check": license_check, "models": MODELS, "source": {"repo": KANRIPO_REPO, "sha": KANRIPO_SHA},
    "results": {f"{m}|{s}": v for (m, s), v in results.items()}, "gap_simp_minus_trad_ctrl_top1": gaps,
    "per_passage": per_passage,
}, open(out / "B_sikubert_fillmask.json", "w"), ensure_ascii=False, indent=1)
print("\nĐã lưu /content/results/B_sikubert_fillmask.json")

## Nhóm C — Benchmark GPU cho kraken<5 + CHAT_models (script gốc)

**Yêu cầu:** `pip install "kraken<5"`, `git clone https://github.com/colibrisson/CHAT_models`, chạy **đúng script gốc** `demo/chat_models_demo.py` trên 2 ảnh trong `demo/`, đo thời gian segmentation trên GPU bằng `time.time()`, so với baseline CPU **459.5 giây/trang**. Tên tham số device phải tra từ `kraken.blla.segment.__doc__` trong chính notebook. Tuỳ chọn: đo tốc độ/epoch của `ketos` trên GPU.

**Cách làm (và lý do):**
1. **Python 3.11 riêng.** Theo metadata PyPI, kraken 4.3.13 (bản 4.x mới nhất) khai `requires_python <=3.11.99` và ghim `torch~=2.0.1`, `numpy~=1.23.0`. Colab mặc định dùng Python mới hơn nên `!pip install "kraken<5"` trực tiếp vào kernel sẽ không cài được bản này. Cell tạo venv Python 3.11 bằng `uv` và cài **đúng spec `"kraken<5"`** vào đó, rồi in ra version thực tế đã cài.
2. **Script gốc không bị sửa.** Script đọc `test/*.png` nhưng repo không có thư mục `test/` (ảnh nằm ở `demo/`), nên cell copy `demo/*.png` sang `test/`, giống lần chạy 27/9. Script được chạy bằng `runpy` qua một wrapper nhỏ. Wrapper chỉ (a) bấm giờ `blla.segment` và vòng `rpred` bằng `time.time()`, và (b) khi chạy GPU thì truyền thêm tham số device, **chỉ khi** signature/docstring của kraken đã cài có tham số đó. File `demo/chat_models_demo.py` giữ nguyên.
3. **Confound của baseline.** Số 459.5 giây/trang được đo với **kraken 7.1.1** trên CPU sandbox cloud (25/9), không phải kraken 4.x, và phần cứng khác. Vì vậy cell in cả tỷ số so với 459.5 **và** (mặc định) chạy lại 1 ảnh trên CPU Colab với cùng kraken<5, để có mốc cùng máy. Muốn nhanh hơn thì đặt `RUN_CPU_SAME_MACHINE = False`.
4. **Dự đoán cần kiểm:** trong kraken 4.x, `blla.segment` chỉ đưa forward của mạng lên device; bước hậu xử lý (vector hoá baseline, dựng polygon) vẫn chạy CPU. Nếu thời gian segmentation trên GPU không giảm nhiều thì nút thắt là hậu xử lý, không phải model. Ảnh đầu tiên chạy trên GPU còn gồm cả thời gian khởi tạo CUDA, nên ảnh thứ 2 phản ánh tốc độ thật hơn.
5. **Đối chứng dương:** output phải chứa các chuỗi đã thấy trong log 27/9 (`巫山高`, `謝眺`, `恭惟某官`), để xác nhận H2 tái lập được trên máy khác.

**Tuỳ chọn `ketos` (chỉ đo tốc độ):** `ketos train` và `ketos segtrain` cần ground truth dạng XML (ALTO/PAGE), không nhận ảnh trần. Cell in `--help` của cả hai lệnh trước, rồi tạo **nhãn giả** (pseudo-label: output của chính model, ghi thành ALTO) cho 2 ảnh demo, sau đó fine-tune 1 epoch và 3 epoch. Hiệu thời gian chia 2 ra chi phí/epoch, đã trừ phần khởi động. Kết quả **chỉ dùng ước lượng tốc độ/effort**, không nói gì về chất lượng. Nếu một lệnh lỗi, lỗi được ghi lại thay vì dừng notebook.

**License:** CHAT_models là CC BY-NC 4.0, chỉ dùng cho nghiên cứu.

In [ ]:
# === Cell C — kraken<5 + CHAT_models: segmentation GPU vs baseline CPU 459.5 s/trang ===
RUN_CPU_SAME_MACHINE = True   # mốc CPU cùng máy Colab (chậm — log 27/9: 81 phút cho 2 ảnh trên Mac)
CPU_MAX_IMAGES = 1
CPU_TIMEOUT_S = 3600
RUN_KETOS = True
KETOS_TIMEOUT_S = 30 * 60
BASELINE_SEG_S = 459.5        # kraken 7.1.1, CPU sandbox cloud, 25/9
ANCHORS = ["巫山高", "謝眺", "恭惟某官"]

import subprocess, sys, os, json, time, shutil, shlex, pathlib, difflib, datetime

WORK = pathlib.Path("/content/kraken_bench"); WORK.mkdir(parents=True, exist_ok=True)
OUT = pathlib.Path("/content/results"); OUT.mkdir(parents=True, exist_ok=True)
VENV = WORK / "venv311"
PY, KETOS = VENV / "bin" / "python", VENV / "bin" / "ketos"
REPO = WORK / "CHAT_models"


def sh(cmd, timeout=None, check=True, cwd=None):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, timeout=timeout, cwd=cwd)
    if r.stdout.strip():
        print(r.stdout[-3000:])
    if r.returncode != 0:
        print(r.stderr[-4000:])
        if check:
            raise RuntimeError(f"lệnh lỗi (exit {r.returncode})")
    return r


# 1) venv Python 3.11 + "kraken<5"
sh(f"{sys.executable} -m pip install -q uv")
if not PY.exists():
    sh(f"{sys.executable} -m uv venv --python 3.11 {VENV}")
sh(f'{sys.executable} -m uv pip install --python {PY} -q "kraken<5"')
freeze = sh(f"{sys.executable} -m uv pip freeze --python {PY}").stdout.splitlines()
kraken_pin = next((l for l in freeze if l.lower().startswith("kraken==")), None)
print("đã cài:", kraken_pin, "|", [l for l in freeze if l.lower().startswith(("torch==", "numpy=="))])
assert kraken_pin and kraken_pin.split("==")[1].startswith("4."), "không cài được kraken 4.x"

# 2) clone CHAT_models + chuẩn bị test/ như lần chạy 27/9
if not REPO.exists():
    sh(f"git clone -q https://github.com/colibrisson/CHAT_models {REPO}")
chat_sha = sh(f"git -C {REPO} rev-parse HEAD").stdout.strip()
print("CHAT_models HEAD:", chat_sha, "| log 27/9 dùng 9b86ab6c…", "(khớp)" if chat_sha.startswith("9b86ab6c") else "(KHÁC — ghi chú khi báo cáo)")
for w in ("chat_seg.mlmodel", "chat_rec.mlmodel"):
    assert (REPO / "models" / w).stat().st_size > 1_000_000, f"{w} quá nhỏ — có thể là pointer LFS"
images = sorted((REPO / "demo").glob("*.png"))
(REPO / "test").mkdir(exist_ok=True)
for p in images:
    shutil.copy(p, REPO / "test" / p.name)
print("ảnh:", [p.name for p in images])

# 3) tra tham số device thật trong kraken đã cài (không đoán)
DOC_SRC = """
import inspect, json
from kraken import blla
from kraken.lib import models
print("=== kraken.blla.segment.__doc__ ===")
print(blla.segment.__doc__)
print("=== kraken.lib.models.load_any.__doc__ ===")
print(models.load_any.__doc__)
print("PARAMS " + json.dumps({"segment": list(inspect.signature(blla.segment).parameters),
                              "load_any": list(inspect.signature(models.load_any).parameters)}))
"""
doc = sh(f"{PY} -W ignore -c {shlex.quote(DOC_SRC)}").stdout
params = json.loads(next(l for l in doc.splitlines() if l.startswith("PARAMS "))[7:])
SEG_HAS_DEVICE = "device" in params["segment"]
LOAD_HAS_DEVICE = "device" in params["load_any"]
gpu_in_venv = sh(f'{PY} -W ignore -c "import torch; print(torch.cuda.is_available())"').stdout.strip().endswith("True")
print(f"blla.segment có tham số 'device': {SEG_HAS_DEVICE} | load_any có 'device': {LOAD_HAS_DEVICE} | GPU thấy trong venv: {gpu_in_venv}")

# 4) wrapper: chạy script gốc bằng runpy, chỉ bấm giờ + (tuỳ chọn) truyền device
WRAP_SRC = """
import os, sys, json, time, runpy
DEVICE = os.environ.get("BENCH_DEVICE", "")
SEG_DEV = os.environ.get("BENCH_SEG_DEVICE") == "1"
LOAD_DEV = os.environ.get("BENCH_LOAD_DEVICE") == "1"
LOG = os.environ["BENCH_LOG"]
import torch
from kraken import blla, rpred
from kraken.lib import models

def sync():
    if DEVICE.startswith("cuda"):
        torch.cuda.synchronize()

def log(**kw):
    with open(LOG, "a") as f:
        f.write(json.dumps(kw) + "\\n")

_seg, _load, _rpred = blla.segment, models.load_any, rpred.rpred

def segment(*a, **k):
    if DEVICE and SEG_DEV:
        k.setdefault("device", DEVICE)
    sync(); t = time.time()
    r = _seg(*a, **k)
    sync()
    log(step="segment", seconds=time.time() - t, n_lines=len(r["lines"]), size=list(a[0].size),
        device=k.get("device", "cpu (mặc định)"))
    return r

def load_any(*a, **k):
    if DEVICE and LOAD_DEV:
        k.setdefault("device", DEVICE)
    return _load(*a, **k)

def rpred_timed(*a, **k):
    gen, total, n = _rpred(*a, **k), 0.0, 0
    while True:
        t = time.time()
        try:
            rec = next(gen)
        except StopIteration:
            break
        sync(); total += time.time() - t; n += 1
        yield rec
    log(step="recognition", seconds=total, n_lines=n)

blla.segment, models.load_any, rpred.rpred = segment, load_any, rpred_timed
t0 = time.time()
runpy.run_path("demo/chat_models_demo.py", run_name="__main__")
log(step="script_total", seconds=time.time() - t0)
"""
(WORK / "wrap.py").write_text(WRAP_SRC)
size2name = {}
from PIL import Image
for p in images:
    with Image.open(p) as im:
        size2name[tuple(im.size)] = p.name


def run_original(tag, device, workdir, timeout=None):
    log = OUT / f"C_steps_{tag}.jsonl"
    log.unlink(missing_ok=True)
    env = dict(os.environ, BENCH_DEVICE=device, BENCH_LOG=str(log),
               BENCH_SEG_DEVICE="1" if SEG_HAS_DEVICE else "0", BENCH_LOAD_DEVICE="1" if LOAD_HAS_DEVICE else "0")
    t = time.time()
    try:
        r = subprocess.run([str(PY), "-W", "ignore", str(WORK / "wrap.py")], cwd=workdir, env=env,
                           capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        print(f"⏱ {tag}: quá {timeout} s")
        return {"tag": tag, "timeout_s": timeout}
    wall = time.time() - t
    (OUT / f"C_stdout_{tag}.txt").write_text(r.stdout)
    print(f"--- {tag}: exit {r.returncode}, {wall:.1f} s ---")
    print(r.stdout[-1500:])
    if r.returncode != 0:
        print(r.stderr[-3000:])
    steps = [json.loads(l) for l in open(log)] if log.exists() else []
    for s in steps:
        if "size" in s:
            s["image"] = size2name.get(tuple(s["size"]), "?")
    return {"tag": tag, "device": device or "cpu", "returncode": r.returncode, "wall_s": wall, "steps": steps, "stdout": r.stdout}


runs = {}
if gpu_in_venv and SEG_HAS_DEVICE:
    runs["gpu"] = run_original("gpu", "cuda:0", REPO)
else:
    print("⚠️ Không đo GPU: " + ("venv không thấy GPU (bật Runtime → GPU)" if not gpu_in_venv else "blla.segment không có tham số 'device'"))

if RUN_CPU_SAME_MACHINE:
    cpu_dir = WORK / "cpu_run"
    shutil.rmtree(cpu_dir, ignore_errors=True)
    (cpu_dir / "test").mkdir(parents=True)
    for sub in ("models", "demo"):
        (cpu_dir / sub).symlink_to(REPO / sub)
    for p in images[:CPU_MAX_IMAGES]:
        shutil.copy(p, cpu_dir / "test" / p.name)
    runs["cpu"] = run_original("cpu", "", cpu_dir, timeout=CPU_TIMEOUT_S)

print("\n### Thời gian script gốc (giây, time.time())\n")
print("| chạy | ảnh | số dòng | segmentation | recognition | seg: baseline 459.5 / đo được |")
print("|---|---|---|---|---|---|")
seg_by = {}
for tag, r in runs.items():
    if "timeout_s" in r:
        print(f"| {tag} | — | — | timeout > {r['timeout_s']} | — | — |")
        continue
    segs = [s for s in r["steps"] if s["step"] == "segment"]
    recs = [s for s in r["steps"] if s["step"] == "recognition"]
    for i, s in enumerate(segs):
        rec_s = recs[i]["seconds"] if i < len(recs) else float("nan")
        seg_by[(tag, s["image"])] = s["seconds"]
        print(f"| {tag} ({s['device']}) | {s['image']} | {s['n_lines']} | {s['seconds']:.1f} | {rec_s:.1f} | ×{BASELINE_SEG_S / s['seconds']:.1f} |")
print("(Ảnh đầu tiên của lượt GPU gồm cả khởi tạo CUDA. Tỷ số với 459.5 lẫn cả khác version kraken lẫn khác phần cứng — so sánh hợp lệ là gpu vs cpu cùng ảnh ở dưới.)")

comparison = {}
for (tag, name), sec in seg_by.items():
    if tag == "cpu" and ("gpu", name) in seg_by:
        comparison[name] = {"seg_cpu_s": sec, "seg_gpu_s": seg_by[("gpu", name)], "speedup": sec / seg_by[("gpu", name)]}
        print(f"Cùng máy, {name}: seg CPU {sec:.1f} s vs GPU {seg_by[('gpu', name)]:.1f} s → ×{sec / seg_by[('gpu', name)]:.1f}")
if "stdout" in runs.get("gpu", {}) and "stdout" in runs.get("cpu", {}):
    comparison["text_similarity_cpu_vs_gpu_first_image_prefix"] = difflib.SequenceMatcher(
        None, runs["cpu"]["stdout"], runs["gpu"]["stdout"][:len(runs["cpu"]["stdout"])]).ratio()

anchors = {}
ref = runs.get("gpu") or runs.get("cpu") or {}
if "stdout" in ref:
    anchors = {a: a in ref["stdout"] for a in ANCHORS}
    print("\nĐối chứng dương (chuỗi log 27/9):", anchors, "→ TÁI LẬP H2" if all(anchors.values()) else "→ thiếu chuỗi mốc — đọc lại output")

# 5) (tuỳ chọn) ketos: tốc độ fine-tune trên GPU bằng nhãn giả
ketos = {}
if RUN_KETOS and gpu_in_venv:
    for sub in ("train", "segtrain"):
        h = sh(f"{KETOS} {sub} --help", check=False)
        (OUT / f"C_ketos_{sub}_help.txt").write_text(h.stdout + h.stderr)
        print(f"=== ketos {sub} --help ===\n{h.stdout[:4000]}")
    GT = WORK / "pseudo_gt"
    GT_SRC = """
import sys, pathlib, shutil, inspect
from PIL import Image
from kraken import blla, rpred, serialization
from kraken.lib import vgsl, models
dev, repo, gt = sys.argv[1], pathlib.Path(sys.argv[2]), pathlib.Path(sys.argv[3])
gt.mkdir(parents=True, exist_ok=True)
print("serialize params:", list(inspect.signature(serialization.serialize).parameters))
seg_model = vgsl.TorchVGSLModel.load_model(str(repo / "models" / "chat_seg.mlmodel"))
rec_model = models.load_any(str(repo / "models" / "chat_rec.mlmodel"), device=dev)
for p in sorted((repo / "demo").glob("*.png")):
    dst = gt / p.name
    shutil.copy(p, dst)
    img = Image.open(dst).convert("L").point(lambda x: 0 if x < 128 else 255, "1")
    seg = blla.segment(img, text_direction="vertical-rl", model=seg_model, device=dev)
    for line in seg["lines"]:
        if line["baseline"][0][1] > line["baseline"][-1][1]:
            line["baseline"].reverse()
    recs = list(rpred.rpred(rec_model, img, seg))
    xml = serialization.serialize(recs, image_name=str(dst), image_size=img.size, writing_mode="vertical-rl",
                                  regions=seg.get("regions"), template="alto")
    dst.with_suffix(".xml").write_text(xml)
    print(dst.with_suffix(".xml"), len(recs), "dòng")
"""
    (WORK / "make_gt.py").write_text(GT_SRC)
    g = sh(f"{PY} -W ignore {WORK / 'make_gt.py'} cuda:0 {REPO} {GT}", check=False)
    ketos["pseudo_gt"] = {"returncode": g.returncode, "stderr_tail": g.stderr[-2000:]}
    xmls = sorted(str(p) for p in GT.glob("*.xml"))
    if g.returncode == 0 and xmls:
        for sub, model in (("segtrain", REPO / "models" / "chat_seg.mlmodel"), ("train", REPO / "models" / "chat_rec.mlmodel")):
            ketos[sub] = {}
            for epochs in (1, 3):
                base = [str(KETOS), sub, "-i", str(model), "-f", "alto", "-d", "cuda:0", "-q", "fixed", "-N", str(epochs),
                        "-o", str(WORK / f"ft_{sub}_{epochs}"), "-e", xmls[0], *xmls]
                attempts = [base, base[:2] + ["--resize", "both"] + base[2:]]
                for cmd in attempts:
                    t = time.time()
                    try:
                        r = subprocess.run(cmd, capture_output=True, text=True, timeout=KETOS_TIMEOUT_S, cwd=WORK)
                        res = {"cmd": " ".join(cmd), "returncode": r.returncode, "wall_s": time.time() - t,
                               "stderr_tail": r.stderr[-2500:], "stdout_tail": r.stdout[-1500:]}
                    except subprocess.TimeoutExpired:
                        res = {"cmd": " ".join(cmd), "timeout_s": KETOS_TIMEOUT_S}
                    print(f"ketos {sub} N={epochs}: exit {res.get('returncode')} sau {res.get('wall_s', float('nan')):.1f} s")
                    if res.get("returncode"):
                        print(res["stderr_tail"][-1200:])
                    ketos[sub][epochs] = res
                    if res.get("returncode") == 0 or "resize" not in (res.get("stderr_tail", "") + res.get("stdout_tail", "")).lower():
                        break
            ok = all(ketos[sub].get(e, {}).get("returncode") == 0 for e in (1, 3))
            if ok:
                per_epoch = (ketos[sub][3]["wall_s"] - ketos[sub][1]["wall_s"]) / 2
                ketos[sub]["per_epoch_s_marginal"] = per_epoch
                print(f"→ ketos {sub}: ~{per_epoch:.1f} s/epoch trên {len(xmls)} trang nhãn giả (đã trừ phần khởi động)")
    else:
        print("Không tạo được nhãn giả — bỏ qua ketos (lỗi đã ghi trong JSON).")

json.dump({"experiment": "C_kraken_chat_original_script_gpu", "timestamp_utc": datetime.datetime.utcnow().isoformat() + "Z",
           "kraken": kraken_pin, "chat_models_sha": chat_sha, "segment_params": params,
           "seg_has_device": SEG_HAS_DEVICE, "load_any_has_device": LOAD_HAS_DEVICE, "gpu_in_venv": gpu_in_venv,
           "baseline": {"seg_s_per_page": BASELINE_SEG_S, "note": "kraken 7.1.1, CPU sandbox cloud 25/9 — khác version/phần cứng"},
           "config": {"cpu_same_machine": RUN_CPU_SAME_MACHINE, "cpu_max_images": CPU_MAX_IMAGES},
           "runs": runs, "comparison": comparison, "anchors": anchors, "ketos": ketos, "venv_freeze": freeze},
          open(OUT / "C_kraken_bench.json", "w"), ensure_ascii=False, indent=1)
print("\nĐã lưu /content/results/C_kraken_bench.json")

In [ ]:
# Cell cuối — nén /content/results để tải về (kèm notebook có output)
import shutil, pathlib
p = pathlib.Path("/content/results")
if p.exists() and any(p.iterdir()):
    z = shutil.make_archive("/content/hannom_model_experiments_results", "zip", p)
    print("Đã tạo", z)
    try:
        from google.colab import files
        files.download(z)
    except ImportError:
        print("Không chạy trên Colab — tự lấy file zip ở trên.")
else:
    print("Chưa có kết quả trong /content/results — chạy Cell A/B/C trước.")